# Case Study 1 — Gradient-Boosted Direction Classifier on US Equities

**Hypothesis:** A gradient-boosted classifier trained on technical, macro, and cross-sectional features can predict the *sign* of next-day US equity returns — and the resulting daily-rebalanced long/short portfolio beats buy-and-hold after costs.

**Why this notebook is different from the typical 'I trained ML on stocks' demo:**

- **Purged + embargoed walk-forward CV** (`backtester.eval.walkforward`) so the label window can't leak into training.
- **Realistic transaction costs** at the bps level (commission + half-spread, `EQUITIES_LIQUID`).
- **Deflated Sharpe Ratio** (Bailey & López de Prado 2014) to account for multiple-trial bias.
- **Bootstrap 95% CI** on Sharpe (block resample, 20-day blocks) — point estimates are vibes.
- **Per-regime breakdown** so a strategy that only works in bulls gets exposed.
- **Point-in-time-aware universe** (`samples/universe_us_liquid.csv`) avoids most survivorship bias.
- **Non-overlapping forecasts**: 1-day labels with daily rebalancing → no autocorrelation inflation.

**Reproducibility:** caches yfinance/FRED downloads under `data_cache/` (date-keyed). Required env: `FRED_API_KEY` (free, https://fred.stlouisfed.org/). Yahoo Finance & Binance need no key.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
load_dotenv(ROOT / '.env', override=True)

from backtester.data.yfinance import fetch_daily as yf_fetch
from backtester.data.fred import fetch_series as fred_fetch
from backtester.data.universe import load_universe
from backtester.features import (
    rolling_volatility, rsi, macd,
    macro_features, momentum_rank, vol_rank,
)
from backtester.eval.walkforward import walk_forward_splits
from backtester.eval.statistics import (
    annualised_sharpe, deflated_sharpe_ratio, bootstrap_ci,
)
from backtester.eval.costs import EQUITIES_LIQUID
from backtester.eval.regimes import trend_regimes, per_regime_metrics
from backtester.strategy import (
    long_short_quantile_weights, daily_returns_from_book, apply_book_costs,
)
from backtester.models import GBMClassifier

pd.options.display.float_format = '{:,.4f}'.format
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True})
RNG = np.random.default_rng(42)

START = '2010-01-01'
END = '2024-12-31'
LABEL_HORIZON = 1
TOP_BOTTOM_QUANTILE = 0.2
N_SPLITS = 6
EMBARGO = 5

## 1. Load price data for the eligible universe

yfinance occasionally times out; loaders retry with exponential backoff. Tickers are eligible from their `first_eligible` date onwards — this rules out (most) survivorship bias from the cross-sectional ranks.

In [ ]:
universe = load_universe()
tickers = sorted(universe['ticker'])
print(f'Universe size: {len(tickers)}')

prices = {}
missing = []
for tkr in tickers:
    try:
        df = yf_fetch(tkr, start=START, end=END)
        prices[tkr] = df['adj_close'] if 'adj_close' in df.columns else df['close']
    except Exception as exc:  # noqa: BLE001 — log and skip flaky downloads
        missing.append((tkr, str(exc)[:80]))

px = pd.DataFrame(prices).sort_index()
px = px.loc[~px.index.duplicated(keep='first')]
print(f'Price panel: {px.shape[0]} rows × {px.shape[1]} tickers, '
      f'{px.index.min().date()} → {px.index.max().date()}')
if missing:
    print(f'\n{len(missing)} tickers missing (re-run cell to retry):')
    for tkr, msg in missing:
        print(f'  {tkr}: {msg}')

## 2. Build the feature panel

Causality is enforced by `tests/test_features_leakage.py`. For each (date, ticker) we stack:

- **Single-asset technical:** 5/20/60-day momentum, 20-day vol, RSI(14), MACD line.
- **Cross-sectional:** momentum rank (12-1), vol rank.
- **Macro (lagged 1d):** VIX level + 5d change, T10Y2Y slope + change, BAA10Y credit spread + change.

In [ ]:
rets = px.pct_change()
log_rets = np.log(px).diff()

tech_blocks = []
for tkr in px.columns:
    s = px[tkr]
    block = pd.DataFrame({
        'mom_5':     s.pct_change(5),
        'mom_20':    s.pct_change(20),
        'mom_60':    s.pct_change(60),
        'vol_20':    rolling_volatility(log_rets[tkr], window=20),
        'rsi_14':    rsi(s, period=14),
        'macd_line': macd(s)['macd_line'],
    })
    block['ticker'] = tkr
    block.index.name = 'datetime'
    tech_blocks.append(block.reset_index())
tech_long = pd.concat(tech_blocks, ignore_index=True).set_index(['datetime', 'ticker']).sort_index()

mom_panel = momentum_rank(px)
vol_panel = vol_rank(rets)
tech_long['mom_rank'] = mom_panel.stack().reindex(tech_long.index).values
tech_long['vol_rank'] = vol_panel.stack().reindex(tech_long.index).values

calendar = px.index
fred_series = {}
for sid in ['VIXCLS', 'T10Y2Y', 'BAA10Y']:
    try:
        fred_series[sid] = fred_fetch(sid)
    except RuntimeError as exc:
        print(f'FRED skipped ({sid}): {exc}')
macro = macro_features(fred_series, calendar) if fred_series else pd.DataFrame(index=calendar)

feats = tech_long.join(macro, on='datetime') if not macro.empty else tech_long
feats = feats.dropna()
print(f'Feature rows after dropna: {len(feats):,}')
print('Feature columns:', list(feats.columns))
feats.head()

## 3. Build the label

**1-day-ahead** return sign. Daily rebalancing means each forecast covers a non-overlapping window — no autocorrelation inflation of the Sharpe ratio.

In [ ]:
fwd_ret = px.pct_change(LABEL_HORIZON).shift(-LABEL_HORIZON)
fwd_long = fwd_ret.stack().rename('fwd_ret')
fwd_long.index.names = ['datetime', 'ticker']
y = (fwd_long > 0).astype(int).rename('y')

design = feats.join(y, how='inner').join(fwd_long, how='inner').dropna()
print(f'Design matrix: {len(design):,} rows')
feature_cols = [c for c in design.columns if c not in {'y', 'fwd_ret'}]

## 4. Walk-forward training

Splits are over the *date axis* (not the row axis) so each fold is a contiguous block of trading days. Purge horizon = `LABEL_HORIZON`. Embargo = 5 trading days.

In [ ]:
dates = design.index.get_level_values('datetime').unique().sort_values()
splits = list(walk_forward_splits(
    n=len(dates), n_splits=N_SPLITS,
    label_horizon=LABEL_HORIZON, embargo=EMBARGO,
))

X = design[feature_cols]
y_full = design['y']

oos_proba = pd.Series(np.nan, index=design.index, dtype=float)
for i, (train_di, test_di) in enumerate(splits, 1):
    train_dates = dates[train_di]
    test_dates = dates[test_di]
    train_mask = design.index.get_level_values('datetime').isin(train_dates)
    test_mask = design.index.get_level_values('datetime').isin(test_dates)
    if train_mask.sum() < 5_000 or test_mask.sum() < 200:
        continue
    model = GBMClassifier()
    model.fit(X[train_mask], y_full[train_mask])
    oos_proba[test_mask] = model.predict_proba(X[test_mask])
    print(f'Fold {i}: train={train_mask.sum():>7,} test={test_mask.sum():>6,} '
          f'| {train_dates.min().date()} → {train_dates.max().date()} '
          f'| OOS {test_dates.min().date()} → {test_dates.max().date()}')

oos = design.loc[oos_proba.dropna().index].copy()
oos['proba'] = oos_proba.dropna()
print(f'\nOOS rows: {len(oos):,}')

## 5. Build the long/short portfolio

Top 20% / bottom 20% of probabilities, equal-weight, dollar-neutral. Costs charged on book-level turnover via `EQUITIES_LIQUID`. All implemented in `backtester.strategy.cross_sectional` and unit-tested.

In [ ]:
weights = long_short_quantile_weights(oos['proba'], quantile=TOP_BOTTOM_QUANTILE)
gross = daily_returns_from_book(weights, oos['fwd_ret'])
net = apply_book_costs(weights, gross, EQUITIES_LIQUID)

n_long = (weights > 0).groupby(level=0).sum().mean()
n_short = (weights < 0).groupby(level=0).sum().mean()
print(f'Avg names per side: long {n_long:.1f}, short {n_short:.1f}')
print(f'Daily returns: {len(net):,} rows, gross mean {gross.mean()*1e4:.2f} bps, '
      f'net mean {net.mean()*1e4:.2f} bps')

## 6. Honest evaluation

`n_trials=20` is a placeholder for the implicit hyperparameter / feature-set candidates explored. Plug in a higher number if you sweep more.

**Verdict criterion:** if `dsr < 0.95`, we have **failed to reject** the null that this strategy is no better than the best of 20 random models — that's the conclusion most signals deserve.

In [ ]:
ret = net.dropna().to_numpy()
sr_gross = annualised_sharpe(gross.dropna().to_numpy())
sr_net = annualised_sharpe(ret)
dsr = deflated_sharpe_ratio(ret, n_trials=20)
ci = bootstrap_ci(ret, statistic=annualised_sharpe, n_resamples=1000, block_size=20, rng=RNG)
ann_ret = (1 + pd.Series(ret).mean()) ** 252 - 1

print(f'Annualised Sharpe (gross):           {sr_gross:>+7.3f}')
print(f'Annualised Sharpe (net of costs):    {sr_net:>+7.3f}')
print(f'Deflated Sharpe Ratio (n_trials=20): {dsr:>7.3f}')
print(f'Bootstrap 95% CI on Sharpe:          [{ci.lower:+.3f}, {ci.upper:+.3f}]')
print(f'Approx annualised return (net):      {ann_ret*100:>+7.2f}%')
print(f'\nVerdict: {"PASSES" if dsr >= 0.95 else "FAILS"} the deflated-Sharpe threshold.')

## 7. Equity curve + drawdown vs. SPY benchmark

If SPY can't be fetched, falls back to an equal-weight average of the universe.

In [ ]:
try:
    spy = yf_fetch('SPY', start=START, end=END)
    bench_close = spy['adj_close'] if 'adj_close' in spy.columns else spy['close']
    bench_label = 'SPY buy & hold'
except RuntimeError:
    bench_close = px.mean(axis=1)
    bench_label = f'Equal-weight univ. ({px.shape[1]} names) — SPY fetch failed'
    print('Falling back to universe-mean benchmark.')

bench_ret = bench_close.pct_change().reindex(net.index).fillna(0.0)
equity_strat = (1 + net).cumprod()
equity_bench = (1 + bench_ret).cumprod()

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(equity_strat, label='GBM long/short (net)', linewidth=1.4)
ax[0].plot(equity_bench, label=bench_label, linewidth=1.0, alpha=0.7)
ax[0].set_ylabel('Equity (×)')
ax[0].legend()
ax[0].set_title('Case 1 — GBM direction classifier on US equities')

running_max = equity_strat.cummax()
dd = equity_strat / running_max - 1
ax[1].fill_between(dd.index, dd.values, 0, color='crimson', alpha=0.4)
ax[1].set_ylabel('Drawdown')
plt.tight_layout()
plt.show()

## 8. Per-regime breakdown

Bull = benchmark above its 200-day SMA. A signal that only works in one regime doesn't generalise.

In [ ]:
regimes = trend_regimes(bench_close, window=200).reindex(net.index)
by_regime = per_regime_metrics(net, regimes, metric=lambda x: annualised_sharpe(x))
pd.Series(by_regime, name='Net Sharpe by regime').to_frame()

## Discussion (fill in with observed numbers from your run)

Re-run end-to-end and write up:

- Did the **deflated Sharpe** clear the ~0.95 threshold?
- Was the bootstrap 95% CI on Sharpe strictly above zero?
- Did the strategy survive both bull and bear regimes?
- How does the equity curve compare to SPY net of costs?

If any answer is "no", that **is** the finding. The whole point of this notebook is to be honest about it — and an honest negative is a better portfolio piece than a fake positive.